# Fine-tune T5-small on ParaDetox and GYAFC together
Train **one** model that both removes hostility and fixes register, selected by the task prefix.

**Why not train them in sequence.** Fine-tuning on one dataset and then the other causes catastrophic forgetting: the second task overwrites the first, and the prefix distinction collapses so both prefixes return the same text. Mixing the datasets instead puts both tasks in *every* batch, so neither can overwrite the other. This is what T5's task prefixes were designed for.

**In Colab:** choose **Runtime - Change runtime type - GPU**, then run the cells in order. Section 3 asks for the GYAFC corpus, which is not on the Hugging Face Hub; ParaDetox is fetched from the Hub automatically.

Set `SMOKE_TEST = True` in the configuration cell for a quick end-to-end check before full training.

GYAFC is distributed under a Yahoo/Grammarly research license. Do not commit the corpus to this repository, and do not redistribute it.

Sources: [ParaDetox](https://huggingface.co/datasets/s-nlp/paradetox), [GYAFC (Rao & Tetreault, 2018)](https://aclanthology.org/N18-1012/), [T5-small](https://huggingface.co/google-t5/t5-small).

## 1. Install dependencies
Use a fresh Colab runtime with Python 3.10 or newer. Colab provides PyTorch; install the remaining packages below. If Colab asks to restart the session, restart it and continue from the next cell.

`sacrebleu` is used only by the BLEU cell in section 10.

In [7]:
%pip install -q "transformers==4.57.3" "datasets==4.4.1" "accelerate==1.12.0" "sentencepiece==0.2.1" "sacrebleu==2.4.3"

## 2. Configure the run
Every row becomes `<prefix><input>` -> `<rewrite>`, and the two prefixes are the only thing telling the model which job to do.

**`MAX_LENGTH` is 128, not 64.** GYAFC alone needed only 64, but ParaDetox comments are longer, and one setting has to cover both.

**`DETOX_UPSAMPLE` controls the mix, and the default is deliberately 1.** GYAFC contributes 104,562 training pairs against ParaDetox's 15,787, a 6.6:1 imbalance, which looks like something that needs correcting. Mostly it does not.

Upsampling does not give the model *more exposure per epoch* to each ParaDetox example - it visits every one of them once per epoch either way. What it changes is how many times each example is repeated, and therefore how fast that task overfits. The single-task ParaDetox run reached its best validation loss at epoch 3 (0.9076) and was already worsening by epoch 4 (0.9083), so three passes is roughly its limit. At `EPOCHS = 3`:

| `DETOX_UPSAMPLE` | passes over ParaDetox | train rows | time on an A100 |
| --- | --- | --- | --- |
| **1 (default)** | 3 - matches the best single-task run | ~120,000 | ~47 min |
| 2 | 6 - past where it started overfitting | ~136,000 | ~53 min |
| 3 | 9 - well past it | ~152,000 | ~60 min |

What upsampling *would* genuinely protect against is capacity competition: a 60M-parameter model divides its weights between the two tasks roughly in proportion to how often it sees them, and detoxification is only about 13% of the mix at `DETOX_UPSAMPLE = 1`. That effect is real but hard to predict, so the honest approach is to measure it rather than pre-compensate for it. Section 9 reports **test loss per task** for exactly this reason.

**Read those two numbers and decide.** A detox test loss near 0.91 means the imbalance did no harm and the default was right. A clearly worse one means the task did lose out - raise `DETOX_UPSAMPLE` to 2 and watch whether detox improves or simply starts overfitting.

The effective training batch is `8 x 2 = 16`, as in the other two notebooks, and training starts from the **base** T5-small rather than from either fine-tuned model.

In [8]:
import os
import random
import shutil
import zipfile
from collections import Counter
from pathlib import Path

import torch
from datasets import Dataset, DatasetDict, concatenate_datasets, load_dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

MODEL_NAME = "google-t5/t5-small"

# --- ParaDetox (Hugging Face Hub) ---
DETOX_DATASET = "s-nlp/paradetox"
DETOX_SOURCE = "en_toxic_comment"
DETOX_TARGET = "en_neutral_comment"
DETOX_PREFIX = "detoxify: "
DETOX_UPSAMPLE = 1  # Repeat ParaDetox this many times in the training mix. See section 2.

# --- GYAFC (local, license-restricted) ---
GYAFC_DIRNAME = "GYAFC_Corpus"
DOMAINS = ("Entertainment_Music", "Family_Relationships")
SPLIT_DIRS = {"train": "train", "validation": "tune", "test": "test"}
GYAFC_SOURCE = "informal"
GYAFC_TARGET = "formal"
GYAFC_PREFIX = "formalize: "
REFERENCE_COUNT = 4          # formal.ref0 .. formal.ref3 in tune/ and test/
LABEL_REFERENCE_INDEX = 0    # Which human reference supplies the loss label.
DROP_TRAIN_OVERLAP = True
DROP_IDENTITY_PAIRS = False

EXTRACT_ROOT = Path("/content") if Path("/content").is_dir() else Path(".")

SEED = 42
MAX_LENGTH = 128
EPOCHS = 3
LEARNING_RATE = 3e-4
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2
SMOKE_TEST = False
SMOKE_STEPS = 5

RUN_DIR = "./both-t5-smoke-checkpoints" if SMOKE_TEST else "./both-t5-checkpoints"
EXPORT_DIR = "./both-t5-smoke" if SMOKE_TEST else "./both-t5"
RESULTS_DIR = f"{EXPORT_DIR}-results"
os.environ["WANDB_DISABLED"] = "true"
set_seed(SEED)
print("CUDA GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available")
print("Mode:", "smoke test" if SMOKE_TEST else "full training")
if not torch.cuda.is_available():
    print("For Colab training, switch to a GPU runtime. CPU training will be much slower.")

CUDA GPU: Tesla T4
Mode: full training


## 3. Provide the GYAFC corpus
GYAFC is license-restricted and is not on the Hugging Face Hub, so `load_dataset("name")` cannot fetch it. **Upload `GYAFC_Corpus.zip` to the Colab session at `/content` before running the cell below.** The cell searches the working directory and `/content` first, extracts the archive, and never prompts for an upload; if it finds nothing it lists every path it checked.

`/content` is wiped when the runtime recycles, so re-upload the archive after a restart. It is about 6.5 MB.

Extraction rejects absolute paths, members that escape the target folder, and symlinks, and it skips the `__MACOSX/` resource forks that the official archive contains.

In [9]:
SEARCH_DIRS = (
    Path("."),            # The Colab working directory is /content.
    Path("/content"),
    Path("/content/data"),
    Path("data"),
    Path("..") / "data",
)

def looks_like_corpus(path):
    """True only if the expected data files are present, not just the folder."""
    return all((path / domain / "train" / "informal").is_file() for domain in DOMAINS)


def safe_members(archive, target):
    """Yield only GYAFC_Corpus/* members, rejecting escaping paths and symlinks."""
    for member in archive.infolist():
        name = member.filename
        if name.startswith("__MACOSX/") or Path(name).name.startswith("._"):
            continue  # macOS resource forks in the official archive.
        if name != f"{GYAFC_DIRNAME}/" and not name.startswith(f"{GYAFC_DIRNAME}/"):
            raise ValueError(f"Unexpected ZIP member outside {GYAFC_DIRNAME}/: {name}")
        if (member.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError(f"Symlink in archive: {name}")
        destination = (EXTRACT_ROOT / name).resolve()
        if destination != target and not destination.is_relative_to(target):
            raise ValueError(f"Unexpected ZIP member path: {name}")
        yield member


def extract_corpus(archive_path):
    target = (EXTRACT_ROOT / GYAFC_DIRNAME).resolve()
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(EXTRACT_ROOT, members=list(safe_members(archive, target)))
    return target


def resolve_corpus():
    for directory in SEARCH_DIRS:
        candidate = directory / GYAFC_DIRNAME
        if looks_like_corpus(candidate):
            return candidate.resolve()
    for directory in SEARCH_DIRS:
        archive = directory / f"{GYAFC_DIRNAME}.zip"
        if archive.is_file():
            print("Extracting", archive.resolve())
            extracted = extract_corpus(archive)
            if looks_like_corpus(extracted):
                return extracted
    searched = "\n  ".join(str(directory.resolve()) for directory in SEARCH_DIRS)
    raise FileNotFoundError(
        f"Could not find {GYAFC_DIRNAME}/ or {GYAFC_DIRNAME}.zip in:\n  {searched}\n"
        f"Upload {GYAFC_DIRNAME}.zip to /content, then re-run this cell."
    )


GYAFC_ROOT = resolve_corpus()
print("Corpus root:", GYAFC_ROOT)
for domain in DOMAINS:
    for directory in SPLIT_DIRS.values():
        assert (GYAFC_ROOT / domain / directory).is_dir(), f"Missing {domain}/{directory}"
print("Found both domains with train/tune/test.")

Extracting /content/GYAFC_Corpus.zip
Corpus root: /content/GYAFC_Corpus
Found both domains with train/tune/test.


## 4. Load GYAFC
Same loader as `train_formal.ipynb`, but each row is emitted with the shared column names this notebook uses throughout: `source` (already carrying its prefix), `target`, `references` and `task`. Unified columns are what let the two corpora be concatenated later.

GYAFC ships its own train/tune/test splits, so `tune` becomes validation. The `tune` and `test` folders are **cross-named**: the source sentences are in `informal`, and their four human formal references are `formal.ref0`-`formal.ref3`. Training rows have only one rewrite each, so their `references` list holds a single entry.

In [10]:
def read_lines(path):
    text = Path(path).read_text(encoding="utf-8")
    lines = text.splitlines()
    # Every GYAFC file ends in a newline and has no interior blank lines. splitlines()
    # also breaks on \x0b \x0c \x85 \u2028 \u2029, so cross-check against newlines.
    assert len(lines) == text.count("\n"), f"{path}: unexpected line breaks or no final newline"
    return lines


def read_gyafc_split(domain, directory):
    base = GYAFC_ROOT / domain / directory
    sources = read_lines(base / GYAFC_SOURCE)
    if directory == "train":
        reference_files = [GYAFC_TARGET]
    else:
        # tune/ and test/ are cross-named: informal is the source, formal.ref* its references.
        reference_files = [f"{GYAFC_TARGET}.ref{index}" for index in range(REFERENCE_COUNT)]
    references = [read_lines(base / name) for name in reference_files]
    for name, column in zip(reference_files, references):
        assert len(column) == len(sources), (
            f"{domain}/{directory}: {name} has {len(column):,} lines, "
            f"but {GYAFC_SOURCE} has {len(sources):,}"
        )
    label_index = 0 if directory == "train" else LABEL_REFERENCE_INDEX
    rows, dropped = [], 0
    for position, source in enumerate(sources):
        source = source.strip()
        candidates = [column[position].strip() for column in references]
        if not source or not all(candidates):
            dropped += 1
            continue
        rows.append({
            "source": GYAFC_PREFIX + source,
            "target": candidates[label_index],
            "references": candidates,
            "task": "formalize",
        })
    print(f"  {domain}/{directory}: {len(rows):,} rows ({dropped} dropped as blank).")
    return rows


def input_key(text):
    return " ".join(text.split()).casefold()


print("GYAFC:")
gyafc_rows = {
    name: [row for domain in DOMAINS for row in read_gyafc_split(domain, directory)]
    for name, directory in SPLIT_DIRS.items()
}

gyafc_train_keys = {input_key(row["source"]) for row in gyafc_rows["train"]}
for name in ("validation", "test"):
    overlapping = sum(1 for row in gyafc_rows[name] if input_key(row["source"]) in gyafc_train_keys)
    print(f"  {name}: {overlapping} of {len(gyafc_rows[name]):,} sources also appear in train.")
    if DROP_TRAIN_OVERLAP:
        gyafc_rows[name] = [
            row for row in gyafc_rows[name] if input_key(row["source"]) not in gyafc_train_keys
        ]

identity = sum(
    1 for row in gyafc_rows["train"] if input_key(row["source"]) == input_key(GYAFC_PREFIX + row["target"])
)
print(f"  train: {identity:,} pairs whose formal side matches the informal side.")
if DROP_IDENTITY_PAIRS:
    gyafc_rows["train"] = [
        row for row in gyafc_rows["train"]
        if input_key(row["source"]) != input_key(GYAFC_PREFIX + row["target"])
    ]
print({name: len(rows) for name, rows in gyafc_rows.items()})

GYAFC:
  Entertainment_Music/train: 52,595 rows (0 dropped as blank).
  Family_Relationships/train: 51,967 rows (0 dropped as blank).
  Entertainment_Music/tune: 2,877 rows (0 dropped as blank).
  Family_Relationships/tune: 2,788 rows (0 dropped as blank).
  Entertainment_Music/test: 1,416 rows (0 dropped as blank).
  Family_Relationships/test: 1,332 rows (0 dropped as blank).
  validation: 10 of 5,665 sources also appear in train.
  test: 27 of 2,748 sources also appear in train.
  train: 1,200 pairs whose formal side matches the informal side.
{'train': 104562, 'validation': 5655, 'test': 2721}


## 5. Load ParaDetox
ParaDetox provides one `train` split on the Hub, so this notebook makes its own 80/10/10 division, exactly as `train_detox.ipynb` does.

Some toxic comments appear more than once with different rewrites. The split is taken over **unique normalised inputs** rather than rows, so a comment and its alternative rewrites cannot land on both sides of the split. Group keys ignore case and extra whitespace; the original text is kept for training.

ParaDetox has one rewrite per row, so its `references` list holds a single entry - the same shape GYAFC's training rows use.

In [11]:
raw = load_dataset(DETOX_DATASET, split="train")
required_columns = {DETOX_SOURCE, DETOX_TARGET}
assert required_columns.issubset(raw.column_names), f"Missing columns: {required_columns - set(raw.column_names)}"


def valid_pair(row):
    return all(isinstance(row[column], str) and row[column].strip() for column in required_columns)


original_count = len(raw)
raw = raw.filter(valid_pair)
print(f"ParaDetox: loaded {original_count:,} rows; retained {len(raw):,} nonempty pairs.")


def grouped_split_indices(texts, seed=42):
    keys = [input_key(text) for text in texts]
    unique_keys = sorted(set(keys))
    if len(unique_keys) < 10:
        raise ValueError("Need at least 10 unique inputs for this split.")
    random.Random(seed).shuffle(unique_keys)
    train_end = int(0.8 * len(unique_keys))
    validation_end = int(0.9 * len(unique_keys))
    group_sets = {
        "train": set(unique_keys[:train_end]),
        "validation": set(unique_keys[train_end:validation_end]),
        "test": set(unique_keys[validation_end:]),
    }
    assert group_sets["train"].isdisjoint(group_sets["validation"])
    assert group_sets["train"].isdisjoint(group_sets["test"])
    assert group_sets["validation"].isdisjoint(group_sets["test"])
    assignments = {key: name for name, group in group_sets.items() for key in group}
    indices = {name: [] for name in group_sets}
    for index, key in enumerate(keys):
        indices[assignments[key]].append(index)
    assert sum(map(len, indices.values())) == len(texts)
    assert all(indices.values())
    return indices


detox_indices = grouped_split_indices(raw[DETOX_SOURCE], SEED)
detox_rows = {}
for name, indices in detox_indices.items():
    subset = raw.select(indices)
    detox_rows[name] = [
        {
            "source": DETOX_PREFIX + source.strip(),
            "target": target.strip(),
            "references": [target.strip()],
            "task": "detoxify",
        }
        for source, target in zip(subset[DETOX_SOURCE], subset[DETOX_TARGET])
    ]
print({name: len(rows) for name, rows in detox_rows.items()})

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.tsv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/19744 [00:00<?, ? examples/s]

Filter:   0%|          | 0/19744 [00:00<?, ? examples/s]

ParaDetox: loaded 19,744 rows; retained 19,744 nonempty pairs.
{'train': 15787, 'validation': 1985, 'test': 1972}


## 6. Mix the two corpora
This is the cell that makes the run multi-task rather than sequential.

`train` is the ParaDetox rows repeated `DETOX_UPSAMPLE` times, concatenated with the GYAFC rows, then **shuffled**. The shuffle is not cosmetic: without it every detox example comes first and every formality example second, which is sequential training wearing a disguise, and the forgetting problem comes straight back.

`validation` mixes both tasks unshuffled - order does not affect evaluation - and is what selects the best checkpoint, so the chosen checkpoint has to be good at both jobs.

**The test splits stay separate.** A blended test loss would hide which task regressed. Section 9 reports one number per task instead.

In [12]:
def to_dataset(rows):
    assert rows, "A split ended up empty."
    return Dataset.from_dict({column: [row[column] for row in rows] for column in rows[0]})


detox_train = to_dataset(detox_rows["train"])
gyafc_train = to_dataset(gyafc_rows["train"])

# Repeat the smaller corpus so both tasks get comparable gradient signal.
train_parts = [detox_train] * DETOX_UPSAMPLE + [gyafc_train]
train_dataset = concatenate_datasets(train_parts).shuffle(seed=SEED)

splits = DatasetDict({
    "train": train_dataset,
    "validation": concatenate_datasets([
        to_dataset(detox_rows["validation"]), to_dataset(gyafc_rows["validation"])
    ]),
    "test_detoxify": to_dataset(detox_rows["test"]),
    "test_formalize": to_dataset(gyafc_rows["test"]),
})

if SMOKE_TEST:
    limits = {"train": 128, "validation": 32, "test_detoxify": 16, "test_formalize": 16}
    splits = DatasetDict({
        name: data.shuffle(seed=SEED).select(range(min(len(data), limits[name])))
        for name, data in splits.items()
    })

print(splits)
print("\ntrain task mix:", dict(Counter(splits["train"]["task"])))
print("validation task mix:", dict(Counter(splits["validation"]["task"])))

# The shuffle must interleave the tasks; if the first rows are all one task, it did not take.
print("\nfirst 8 training tasks:", splits["train"]["task"][:8])
assert len(set(splits["train"]["task"][:200])) == 2, "Training set is not interleaved - check the shuffle."

for name in ("train", "validation"):
    print(f"\n{name} example source:", splits[name][0]["source"])
    print(f"{name} example target:", splits[name][0]["target"])

DatasetDict({
    train: Dataset({
        features: ['source', 'target', 'references', 'task'],
        num_rows: 120349
    })
    validation: Dataset({
        features: ['source', 'target', 'references', 'task'],
        num_rows: 7640
    })
    test_detoxify: Dataset({
        features: ['source', 'target', 'references', 'task'],
        num_rows: 1972
    })
    test_formalize: Dataset({
        features: ['source', 'target', 'references', 'task'],
        num_rows: 2721
    })
})

train task mix: {'formalize': 104562, 'detoxify': 15787}
validation task mix: {'detoxify': 1985, 'formalize': 5655}

first 8 training tasks: ['formalize', 'detoxify', 'formalize', 'formalize', 'formalize', 'formalize', 'formalize', 'formalize']

train example source: formalize: MC Hammer - You Can't Touch This
train example target: MC Hammer had a hit song called You Can't Touch This.

validation example source: detoxify: i hope they beat each other silly .
validation example target: I think they beat

## 7. Tokenize
The prefix is already part of `source`, so tokenizing is the same for both tasks. Inputs and targets are tokenized separately and truncated to 128 tokens. Nothing is padded here - the collator pads each batch to its own longest sequence and uses `-100` for target padding, which the loss ignores.

In [13]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
assert all(parameter.requires_grad for parameter in model.parameters())

def tokenize_batch(batch):
    encoded = tokenizer(batch["source"], max_length=MAX_LENGTH, truncation=True)
    targets = tokenizer(text_target=batch["target"], max_length=MAX_LENGTH, truncation=True)
    encoded["labels"] = targets["input_ids"]
    return encoded

tokenized = splits.map(tokenize_batch, batched=True, remove_columns=splits["train"].column_names)
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=-100)
print("Trainable parameters:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("Tokenized fields:", tokenized["train"].column_names)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

...

Map:   0%|          | 0/120349 [00:00<?, ? examples/s]

Map:   0%|          | 0/7640 [00:00<?, ? examples/s]

Map:   0%|          | 0/1972 [00:00<?, ? examples/s]

Map:   0%|          | 0/2721 [00:00<?, ? examples/s]

Trainable parameters: 60,506,624
Tokenized fields: ['input_ids', 'attention_mask', 'labels']


## 8. Train
Same settings as the two single-task notebooks, so the comparison stays fair. Evaluate and save after every epoch; the lowest validation loss picks the checkpoint, which is loaded automatically at the end.

At the default `DETOX_UPSAMPLE = 1` the training set is about 120,000 rows, which at an effective batch of 16 is roughly 7,500 optimizer steps per epoch - about 22,600 steps over three epochs, or around 47 minutes on an A100. Raising `DETOX_UPSAMPLE` scales both figures; section 2 has the table.

In [14]:
training_args = Seq2SeqTrainingArguments(
    output_dir=RUN_DIR,
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    eval_strategy="steps" if SMOKE_TEST else "epoch",
    save_strategy="steps" if SMOKE_TEST else "epoch",
    eval_steps=SMOKE_STEPS if SMOKE_TEST else None,
    save_steps=SMOKE_STEPS if SMOKE_TEST else 500,
    max_steps=SMOKE_STEPS if SMOKE_TEST else -1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    logging_steps=1 if SMOKE_TEST else 50,
    fp16=False,
    bf16=False,
    optim="adamw_torch",
    report_to="none",
    push_to_hub=False,
    seed=SEED,
    data_seed=SEED,
)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=collator,
)
train_result = trainer.train()
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation loss:", trainer.state.best_metric)

Epoch,Training Loss,Validation Loss
1,1.514300,1.206992
2,1.455200,1.167515
3,1.376800,1.158461


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Best checkpoint: ./both-t5-checkpoints/checkpoint-22566
Best validation loss: 1.1584614515304565


## 9. Evaluate each task separately
One blended test loss would hide a task that regressed, so each test split is scored on its own. Compare these against the single-task runs: ParaDetox alone reached a test loss near 0.91, GYAFC alone near 1.16. Multi-task numbers close to those mean the model kept both skills. A detox loss clearly above 0.91 is the signal that the 6.6:1 data imbalance cost that task something - raise `DETOX_UPSAMPLE` to 2 and compare. If detox does not improve, the imbalance was not the problem and the extra passes are only overfitting it.

The two columns are still not comparable with each other - different corpora, different reference counts.

In [15]:
all_metrics = {}
for name in ("test_detoxify", "test_formalize"):
    metrics = trainer.evaluate(eval_dataset=tokenized[name], metric_key_prefix=name)
    all_metrics.update(metrics)
    print(f"{name}: loss {metrics[f'{name}_loss']:.4f}  ({len(tokenized[name]):,} rows)")

trainer.save_metrics("train", train_result.metrics)
trainer.save_metrics("test", all_metrics)

# Copy the metric JSONs somewhere small so they can be archived without the checkpoints.
results_dir = Path(RESULTS_DIR)
results_dir.mkdir(parents=True, exist_ok=True)
for name in ("all_results.json", "train_results.json", "test_results.json"):
    source = Path(RUN_DIR) / name
    if source.is_file():
        shutil.copy2(source, results_dir / name)
best_checkpoint = trainer.state.best_model_checkpoint
state_file = Path(best_checkpoint) / "trainer_state.json" if best_checkpoint else Path(RUN_DIR) / "trainer_state.json"
if state_file.is_file():
    shutil.copy2(state_file, results_dir / "trainer_state.json")
print("\nMetrics folder:", results_dir.resolve())
print("Files:", sorted(path.name for path in results_dir.iterdir()))

print("\nValidation loss by epoch (both tasks combined):")
for entry in trainer.state.log_history:
    if "eval_loss" in entry:
        print(f"  epoch {entry['epoch']:.0f}  eval_loss {entry['eval_loss']:.4f}")
print("Best:", trainer.state.best_metric, "at", best_checkpoint)
if SMOKE_TEST:
    print("Smoke-test results only: these are not full-training quality measurements.")

test_detoxify: loss 0.8980  (1,972 rows)
test_formalize: loss 1.1588  (2,721 rows)

Metrics folder: /content/both-t5-results
Files: ['all_results.json', 'test_results.json', 'train_results.json', 'trainer_state.json']

Validation loss by epoch (both tasks combined):
  epoch 1  eval_loss 1.2070
  epoch 2  eval_loss 1.1675
  epoch 3  eval_loss 1.1585
Best: 1.1584614515304565 at ./both-t5-checkpoints/checkpoint-22566


## 10. Save, reload, and check both prefixes
Export the best model and reload it without the Trainer. `rewrite()` takes the prefix explicitly, because a single model now answers to two.

**The test that matters** is at the bottom of this cell: the same sentence under both prefixes must produce *different* output. If the two agree, the prefixes have collapsed into one behaviour and the multi-task mix did not work - which is exactly the failure that sequential training produces.

In [16]:
trainer.save_model(EXPORT_DIR)
tokenizer.save_pretrained(EXPORT_DIR)

# Reload on CPU to avoid holding a second model copy in GPU memory.
inference_tokenizer = AutoTokenizer.from_pretrained(EXPORT_DIR)
inference_model = AutoModelForSeq2SeqLM.from_pretrained(EXPORT_DIR).to("cpu")
inference_model.eval()

@torch.inference_mode()
def rewrite(prefix, text):
    if not isinstance(text, str):
        raise TypeError("text must be a string")
    if not text.strip():
        return ""
    inputs = inference_tokenizer(
        prefix + text, return_tensors="pt", max_length=MAX_LENGTH, truncation=True
    ).to(inference_model.device)
    output_ids = inference_model.generate(
        **inputs, max_new_tokens=MAX_LENGTH, num_beams=4, do_sample=False
    )
    return inference_tokenizer.decode(output_ids[0], skip_special_tokens=True)

detoxify = lambda text: rewrite(DETOX_PREFIX, text)
formalize = lambda text: rewrite(GYAFC_PREFIX, text)
assert detoxify("   ") == ""
print("Saved and reloaded:", Path(EXPORT_DIR).resolve())

probe = "this is the dumbest plan ive heard tbh"
detox_out, formal_out = detoxify(probe), formalize(probe)
print("\nPrefix separation check")
print("  input          :", probe)
print("  detoxify:      :", detox_out)
print("  formalize:     :", formal_out)
print("  prefixes differ:", detox_out != formal_out,
      "" if detox_out != formal_out else " <- PREFIXES COLLAPSED, the mix did not work")

Saved and reloaded: /content/both-t5

Prefix separation check
  input          : this is the dumbest plan ive heard tbh
  detoxify:      : this is the worst plan i have heard tbh
  formalize:     : This is the worst plan I have heard.
  prefixes differ: True 


## 11. Inspect rewrites from both tasks
Five held-out examples per task, each shown with its reference and the model's output. GYAFC rows carry four references; ParaDetox rows carry one.

In [17]:
for split_name, prefix, label in (
    ("test_detoxify", DETOX_PREFIX, "detoxify"),
    ("test_formalize", GYAFC_PREFIX, "formalize"),
):
    print("=" * 72)
    print(label)
    print("=" * 72)
    for example in splits[split_name].select(range(min(5, len(splits[split_name])))):
        source = example["source"][len(prefix):]  # Strip the prefix for display.
        print("INPUT: ", source)
        for index, reference in enumerate(example["references"]):
            print(f"REF{index}:  ", reference)
        print("MODEL: ", rewrite(prefix, source))
        print()

print("Your examples:")
print("  detoxify: ", detoxify("this is a stupid idea and you never listen"))
print("  formalize:", formalize("gonna need that report asap"))

detoxify
INPUT:  i want you all to myself , you sexy beast .
REF0:   i want you all to myself , honey .
MODEL:  i want you all to myself, you sexy beast.

INPUT:  if any of the cry babies who dont respect democracy come , it will be the stupid drain
REF0:   If any of the cry babies who dont respect democracy come , it will be the bad drain.
MODEL:  if any of the cry babies who dont respect democracy come, it will be the drain

INPUT:  time to pull this rubbish .
REF0:   Time to attend to this issue.
MODEL:  Time to pull this.

INPUT:  the election of van der bellen was a victory for stupidity and cultural suicide .
REF0:   the election of van der bellen was a victory for being unknowledgeable and cultural suicide .
MODEL:  the election of van der bellen was a victory for cultural suicide.

INPUT:  clinton is a loser , twice .
REF0:   Clinton lost twice.
MODEL:  clinton is a loser, twice.

formalize
INPUT:  Is Any Baby Really A Freak.
REF0:   Is any baby really a freak.
REF1:   Is any b

## 12. Multi-reference BLEU on the GYAFC test split
GYAFC's standard metric, kept out of the Trainer so beam search does not run every epoch. Compare the score against the single-task `formalize-t5` run to see what multi-task training cost or gained on this task.

**Read the copy baseline first.** It is the BLEU of the unchanged informal input against the same four references. GYAFC's references stay close to their sources, so echoing the input already scores high; a model score near the copy baseline means the model is barely editing.

In [18]:
import sacrebleu

GENERATION_BATCH_SIZE = 64

@torch.inference_mode()
def generate_all(prefixed_texts):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    generator = inference_model.to(device)
    outputs = []
    for start in range(0, len(prefixed_texts), GENERATION_BATCH_SIZE):
        chunk = prefixed_texts[start:start + GENERATION_BATCH_SIZE]
        inputs = inference_tokenizer(
            chunk, return_tensors="pt", padding=True, max_length=MAX_LENGTH, truncation=True
        ).to(device)
        generated = generator.generate(
            **inputs, max_new_tokens=MAX_LENGTH, num_beams=4, do_sample=False
        )
        outputs += inference_tokenizer.batch_decode(generated, skip_special_tokens=True)
        print(f"  generated {min(start + GENERATION_BATCH_SIZE, len(prefixed_texts)):,}/{len(prefixed_texts):,}", flush=True)
    inference_model.to("cpu")
    return [text.strip() for text in outputs]

formal_test = splits["test_formalize"]
prefixed = list(formal_test["source"])
plain = [text[len(GYAFC_PREFIX):] for text in prefixed]
reference_streams = [
    [row[index] for row in formal_test["references"]] for index in range(REFERENCE_COUNT)
]
predictions = generate_all(prefixed)

model_bleu = sacrebleu.corpus_bleu(predictions, reference_streams)
copy_bleu = sacrebleu.corpus_bleu(plain, reference_streams)
unchanged = sum(1 for source, prediction in zip(plain, predictions) if source == prediction)

print(f"\nTest sentences:      {len(plain):,}")
print(f"Model BLEU (4 refs): {model_bleu.score:.2f}")
print(f"Copy BLEU (4 refs):  {copy_bleu.score:.2f}   <- echoing the input scores this")
print(f"BLEU over copy:      {model_bleu.score - copy_bleu.score:+.2f}")
print(f"Outputs identical to input: {unchanged:,} ({unchanged / len(plain):.1%})")
if SMOKE_TEST:
    print("Smoke-test results only: these are not full-training quality measurements.")

  generated 64/2,721
  generated 128/2,721
  generated 192/2,721
  generated 256/2,721
  generated 320/2,721
  generated 384/2,721
  generated 448/2,721
  generated 512/2,721
  generated 576/2,721
  generated 640/2,721
  generated 704/2,721
  generated 768/2,721
  generated 832/2,721
  generated 896/2,721
  generated 960/2,721
  generated 1,024/2,721
  generated 1,088/2,721
  generated 1,152/2,721
  generated 1,216/2,721
  generated 1,280/2,721
  generated 1,344/2,721
  generated 1,408/2,721
  generated 1,472/2,721
  generated 1,536/2,721
  generated 1,600/2,721
  generated 1,664/2,721
  generated 1,728/2,721
  generated 1,792/2,721
  generated 1,856/2,721
  generated 1,920/2,721
  generated 1,984/2,721
  generated 2,048/2,721
  generated 2,112/2,721
  generated 2,176/2,721
  generated 2,240/2,721
  generated 2,304/2,721
  generated 2,368/2,721
  generated 2,432/2,721
  generated 2,496/2,721
  generated 2,560/2,721
  generated 2,624/2,721
  generated 2,688/2,721
  generated 2,721/2,721

## 13. Download the results and the model
Two archives, smallest first. `RESULTS_DIR` is a few kilobytes and always transfers; `EXPORT_DIR` is about 225 MB, under the limit below.

**Do not archive `RUN_DIR`.** It holds two checkpoints carrying optimizer state as well as weights, so it runs past a gigabyte. The VS Code Jupyter path tops out near 384 MB per file, because the bytes arrive base64-encoded in a JSON message and the extension buffers them as one JavaScript string, which caps at 2^29 bytes. Colab's own browser file browser streams over HTTPS and has no such ceiling.

Keep **both** prefixes with the model: `detoxify: ` and `formalize: `. They are learned task tokens, and the model is useless without the exact strings.

In [19]:
def archive_and_download(directory):
    """Zip one folder and hand it to the browser. Returns the archive path."""
    path = Path(directory).resolve()
    if not path.is_dir():
        print("Skipping", path, "- not found.")
        return None
    archive_path = shutil.make_archive(str(path), "zip", root_dir=path.parent, base_dir=path.name)
    size_mb = Path(archive_path).stat().st_size / 1_000_000
    print(f"{archive_path}  ({size_mb:,.1f} MB)")
    if size_mb > 384:
        print("  Over the ~384 MB transfer limit; fetch it from Colab's file browser instead.")
    try:
        from google.colab import files
    except ImportError:
        print("  Outside Colab: copy the ZIP from the path above.")
    else:
        files.download(archive_path)
    return archive_path

archive_and_download(RESULTS_DIR)   # Kilobytes: the metrics.
archive_and_download(EXPORT_DIR)    # ~225 MB: the model.

/content/both-t5-results.zip  (0.0 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

/content/both-t5.zip  (225.3 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'/content/both-t5.zip'